# 06 — Self-Query Retriever

In this notebook we will:
1. Understand what a SelfQueryRetriever does and why it matters
2. Define our collection's metadata schema for the LLM
3. Let the LLM automatically extract filters from natural language queries
4. Compare manual filtering vs. self-query filtering
5. Build a full RAG chain with automatic filtering

### Key concept
In notebook 03, we manually wrote `where` filters like `{"quarter": "Q4-2025"}`. That works, but it requires the user to know the exact field names and values.

A **SelfQueryRetriever** uses the LLM to parse the user's question and automatically extract:
- The **semantic query** (what to search for by meaning)
- The **metadata filters** (what to filter by)

Example: *"What did the CFO say about margins in Q3 2025?"*
- Semantic query → `"margins"`
- Filters → `role = "Chief Financial Officer"` AND `quarter = "Q3-2025"`

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../.env")
print("Environment loaded")

Environment loaded


In [2]:
# Connect to ChromaDB through LangChain's Chroma wrapper
# (SelfQueryRetriever needs a LangChain vectorstore, not raw chromadb)
embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="earnings_calls",
    embedding_function=embedding_fn,
    persist_directory="../chroma_db",
)

print(f"Connected to collection with {vectorstore._collection.count()} documents")

/var/folders/9n/86zg9zb12pg62l57ttyq_4pm0000gp/T/ipykernel_75410/3113634356.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/9n/86zg9zb12pg62l57ttyq_4pm0000gp/T/ipykernel_75410/3113634356.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Connected to collection with 176 documents


In [3]:
# Connect to Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)
print("LLM connected")

LLM connected


## Step 1: Define the metadata schema

The SelfQueryRetriever needs to know what metadata fields exist and what values they can take. This is how the LLM knows it can filter by `quarter` or `company`.

---
### Your turn!

Define the `metadata_field_info` list. Each entry describes one filterable field in your ChromaDB collection.

Think about what fields you stored in each document's metadata in notebook 05, and what type each one is.

In [6]:
company = AttributeInfo(
    name="company",
    description="Stock ticker symbol of the company. Values: AAPL (Apple), MSFT (Microsoft)",
    type="string",
)
quarter = AttributeInfo(
    name="quarter",
    description="Fiscal quarter and year in format Q#-YYYY. Values: Q3-2025, Q4-2025, Q1-2026",
    type="string",
)
speaker = AttributeInfo(
    name="speaker",
    description="Full name of the person speaking. Examples: Timothy D. Cook, Kevan Parekh, Satya Nadella",
    type="string",
)
role = AttributeInfo(
    name="role",
    description="Job title of the speaker. Examples: Chief Executive Officer, Chief Financial Officer. Empty string for analysts.",
    type="string",
)

metadata_field_info = [
    company, quarter, speaker, role
]

print(f"Defined {len(metadata_field_info)} metadata fields:")
for f in metadata_field_info:
    print(f"  {f.name} ({f.type}): {f.description[:80]}")

Defined 4 metadata fields:
  company (string): Stock ticker symbol of the company. Values: AAPL (Apple), MSFT (Microsoft)
  quarter (string): Fiscal quarter and year in format Q#-YYYY. Values: Q3-2025, Q4-2025, Q1-2026
  speaker (string): Full name of the person speaking. Examples: Timothy D. Cook, Kevan Parekh, Satya
  role (string): Job title of the speaker. Examples: Chief Executive Officer, Chief Financial Off


## Step 2: Create the SelfQueryRetriever

This wires together the LLM, the vectorstore, and the metadata schema.

In [8]:
document_content_description = "Transcripts of quarterly earnings call presentations and Q&A sessions from public companies"

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,  # shows the extracted query + filters
)

print("SelfQueryRetriever created")

SelfQueryRetriever created


## Step 3: Test automatic filtering

Let's see the SelfQueryRetriever in action. Watch the verbose output — it shows what the LLM extracted as the semantic query and the metadata filters.

In [9]:
def show_retriever_results(docs, max_text=200):
    """Display documents returned by the retriever."""
    print(f"Retrieved {len(docs)} documents:\n")
    for i, doc in enumerate(docs):
        meta = doc.metadata
        print(f"--- Result {i+1} ---")
        print(f"  Company: {meta.get('company')} | Quarter: {meta.get('quarter')} | Speaker: {meta.get('speaker')} | Role: {meta.get('role')}")
        print(f"  Text: {doc.page_content[:max_text]}...")
        print()

In [10]:
# Query 1: The LLM should extract quarter filter
print("Query: 'What were Apple's revenue results in Q4 2025?'\n")
docs = retriever.invoke("What were Apple's revenue results in Q4 2025?")
show_retriever_results(docs)

Query: 'What were Apple's revenue results in Q4 2025?'

Retrieved 4 documents:

--- Result 1 ---
  Company: AAPL | Quarter: Q4-2025 | Speaker: Timothy Cook | Role: 
  Text: Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Today, Apple is proud to report $102.5 billion in revenue, up 8% from a year ago and a September quarter record. Service...

--- Result 2 ---
  Company: AAPL | Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Thanks, Tim, and good afternoon, everyone. Our revenue of $102.5 billion was up 8% year-over-year and is a new September quarter record. We set some temporal quarter records in the Americas, Europe, J...

--- Result 3 ---
  Company: AAPL | Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Okay. Yes, there was no tax-related impact. And what I would say is our strong performance for the quarter is really organically driven. And again, just to reiterate, we had an all-time

In [15]:
# Query 2: The LLM should extract company filter
print("Query: 'What is Microsoft's cloud revenue growth?'\n")
docs = retriever.invoke("What is Microsoft's cloud revenue growth?")
show_retriever_results(docs)

Query: 'What is Microsoft's cloud revenue growth?'

Retrieved 4 documents:

--- Result 1 ---
  Company: MSFT | Quarter: Q1-2026 | Speaker: Satya Nadella | Role: Chairman and Chief Executive Officer
  Text: Thank you, Jonathan. It was a very strong start to our fiscal year. Microsoft Corporation Cloud revenue surpassed $49 billion, up 26% year over year, and our commercial RPO grew over 50% to nearly $40...

--- Result 2 ---
  Company: MSFT | Quarter: Q1-2026 | Speaker: Brad Zelnick | Role: 
  Text: Great. Thanks so much for taking the question. I'll echo my congrats on an amazing start to the year. Amy, is there any way to quantify or frame the revenue impact of Azure being short on capacity? Wh...

--- Result 3 ---
  Company: MSFT | Quarter: Q1-2026 | Speaker: Amy E. Hood | Role: Chief Financial Officer
  Text: Yes, Brent, it's a great question. It's always hard to quantify precisely what would have been the revenue impact in the quarter. But I would offer a way to think about it is A

In [ ]:
# Query 3: The LLM should extract speaker/role filter
print("Query: 'What did the CFO say about gross margins?'\n")
docs = retriever.invoke("What did the CFO say about gross margins?")
show_retriever_results(docs)

## Step 4: Compare manual vs. self-query

Let's see how the SelfQueryRetriever compares to manual filtering from notebook 03.

In [16]:
query = "What did Apple's CFO say about services revenue in Q3 2025?"

# Self-query (automatic)
print("SELF-QUERY (automatic filtering):")
print("=" * 50)
docs_auto = retriever.invoke(query)
show_retriever_results(docs_auto, max_text=150)

# Manual query (hardcoded filters)
print("\nMANUAL (hardcoded filtering):")
print("=" * 50)
docs_manual = vectorstore.similarity_search(
    "services revenue",
    k=4,
    filter={
        "$and": [
            {"company": "AAPL"},
            {"quarter": "Q3-2025"},
        ]
    },
)
show_retriever_results(docs_manual, max_text=150)

SELF-QUERY (automatic filtering):
Retrieved 4 documents:

--- Result 1 ---
  Company: AAPL | Quarter: Q3-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Thanks, Tim, and good afternoon, everyone. Our revenue of $94 billion was up 10% year-over-year and is a new June quarter record. We grew in every geo...

--- Result 2 ---
  Company: AAPL | Quarter: Q3-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Yes, this is Kevan. Let me take that one. In general, I think just reminding, I think you just said, we had a very strong Services quarter, we had an ...

--- Result 3 ---
  Company: AAPL | Quarter: Q3-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Great, Aaron. Thanks for the questions. Let me answer the first one, first around foreign exchange. For Q3, we really had no impact from a foreign exc...

--- Result 4 ---
  Company: AAPL | Quarter: Q3-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Yes, Mike, it

## Step 5: Full RAG chain with self-query

Now let's combine the SelfQueryRetriever with the LLM to build a complete RAG pipeline that automatically filters.

In [24]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an analyst assistant that answers questions about finance using the earnings calls provided.
Use ONLY the provided context to answer. If the context doesn't contain 
the answer, say so. Always mention which company and quarter the information 
comes from.

Context:
{context}

Question: {question}
""")

chain = RAG_PROMPT | llm | StrOutputParser()

def ask(question: str) -> str:
    """RAG pipeline with automatic metadata filtering."""
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('company')} {d.metadata.get('quarter')} — {d.metadata.get('speaker', '?')}]\n{d.page_content}"
        for d in docs
    )
    return chain.invoke({"context": context, "question": question})

print("RAG chain with self-query ready.")

RAG chain with self-query ready.


In [25]:
# Test the full pipeline
q = "What were Apple's total revenue results in Q4 2025 and how did they compare to the previous year?"
print(f"Q: {q}\n")
print(ask(q))

Q: What were Apple's total revenue results in Q4 2025 and how did they compare to the previous year?

According to the earnings call of AAPL Q4-2025, Apple's total revenue was $102.5 billion, which is up 8% year-over-year. This is a new September quarter record for the company.


In [23]:
# Cross-company question
q = "How does Microsoft's cloud business compare to Apple's services business?"
print(f"Q: {q}\n")
print(ask(q))

Q: How does Microsoft's cloud business compare to Apple's services business?

Microsoft's cloud revenue surpassed $49 billion, up 26% year over year, as mentioned in the MSFT Q1-2026 earnings call. 

In contrast, Apple's services business had an all-time record revenue of $27.4 billion, up 13%, as mentioned in the AAPL Q3-2025 earnings call. 

It's worth noting that the information about Apple's services business comes from a different quarter (Q3-2025) compared to Microsoft's cloud business (Q1-2026), so a direct comparison may not be entirely accurate. However, based on the provided context, Microsoft's cloud revenue appears to be significantly higher than Apple's services revenue.


In [ ]:
# No information question
q = "How does Tesla's business perform Q4-2025?"
print(f"Q: {q}\n")
print(ask(q))

Q: How does Tesla's business perform Q4-2025?

The provided context does not contain any information about Tesla's business performance in Q4-2025. Therefore, I cannot provide an answer. The context is empty.


## Summary

In this notebook you learned:
- SelfQueryRetriever uses the LLM to extract metadata filters from natural language
- You need to describe your metadata schema (AttributeInfo) so the LLM knows what's filterable
- The LLM splits the question into a semantic query + structured filters automatically
- This enables natural questions like *"What did Apple's CFO say about margins in Q3?"* without the user knowing the schema

**Next step:** In notebook 07 we'll tackle temporal comparisons — *"How has revenue guidance changed across quarters?"*